In [18]:
%pip install pandas odfpy openpyxl statsmodels scikit-learn prophet matplotlib

Note: you may need to restart the kernel to use updated packages.


In [19]:
# ============================================================
# SETTINGS AND IMPORTS
# ============================================================
import os, json, itertools, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
from prophet import Prophet

warnings.filterwarnings('ignore')
plt.rcParams.update({'font.size': 10, 'figure.dpi': 150,
                     'axes.grid': True, 'grid.alpha': 0.3})
DATA_DIR = '.'                                    # folder containing the .ods files

OUT_DIR = os.path.join(os.getcwd(), 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)
print("Data folder:  ", os.path.abspath(DATA_DIR))
print("Output folder:", OUT_DIR)

FILES = {
    'PORT0201': f'{DATA_DIR}/port0201.ods',
    'PORT0301': f'{DATA_DIR}/port0301.ods',
    'PORT0502': f'{DATA_DIR}/port0502.ods',
}
BREXIT_YEAR = 2021   # end of transition period: 1 January 2021
COVID_YEAR = 2020    # acute pandemic shock year

def num(s):
    """Convert a column to numeric, coercing errors to NaN."""
    return pd.to_numeric(s, errors='coerce')

Data folder:   /Users/rachelnguyen/thesis
Output folder: /Users/rachelnguyen/thesis/outputs


In [20]:
# ---- Download the six DfT tables from GOV.UK ----
import urllib.request

URLS = {
    'port0201.ods': 'https://assets.publishing.service.gov.uk/media/6a622055abcde513b38b6de8/port0201.ods',
    'port0301.ods': 'https://assets.publishing.service.gov.uk/media/6a6220bfe2d191f0bc8b6dfb/port0301.ods',
    'port0502.ods': 'https://assets.publishing.service.gov.uk/media/6a26c86356960b0542c0b19a/port0502.ods',
}

for filename, url in URLS.items():
    if not os.path.exists(filename):
        print('Downloading', filename, '...')
        urllib.request.urlretrieve(url, filename)
    print(filename, '-', os.path.getsize(filename), 'bytes')


port0201.ods - 284041 bytes
port0301.ods - 2569525 bytes
port0502.ods - 246234 bytes


In [21]:
# ============================================================
# PART 1: DATA UNDERSTANDING
# ============================================================
print("="*50)
print("PART 1: DATA UNDERSTANDING")
print("="*50)

# Read sheet 'Data' (header row at index 3) and sheet 'Tonnage_(Both_Directions)' (header row at index 6)
p0301_raw = pd.read_excel(FILES['PORT0301'], sheet_name='Data', engine='odf', skiprows=3, header=0)
p0201_raw = pd.read_excel(FILES['PORT0201'], sheet_name='Data', engine='odf', skiprows=3, header=0)
p0502_raw = pd.read_excel(FILES['PORT0502'], sheet_name='Tonnage_(Both_Directions)', engine='odf', skiprows=6, header=0)

print(f"PORT0301 raw: {p0301_raw.shape[0]} observations")
print(f"PORT0201 raw: {p0201_raw.shape[0]} observations")
print(f"PORT0502 raw: {p0502_raw.shape[0]} ports (across {p0502_raw.shape[1]} time columns)")

PART 1: DATA UNDERSTANDING
PORT0301 raw: 48420 observations
PORT0201 raw: 4463 observations
PORT0502 raw: 54 ports (across 74 time columns)


In [22]:
# ============================================================
# PART 2: DATA PREPARATION
# ============================================================
print("\n" + "="*50)
print("PART 2: DATA PREPARATION")
print("="*50)

# --- Prepare PORT0301 ---
# (Note: the Data sheet of 0301 uses 'Cargo Group Name')
p0301_clean = p0301_raw.dropna(subset=['Cargo Group Name']).copy()
p0301_clean['Tonnage'] = num(p0301_clean['Tonnage'])
p0301_clean = p0301_clean.dropna(subset=['Tonnage'])
p0301_clean = p0301_clean[(p0301_clean['Year'] >= 2009) & (p0301_clean['Year'] <= 2025)]

# --- Prepare PORT0201 ---
# (Note: the Data sheet of 0201 uses 'CargoCode')
p0201_clean = p0201_raw.dropna(subset=['CargoCode']).copy()
p0201_clean['Tonnage'] = num(p0201_clean['Tonnage'])
p0201_clean = p0201_clean.dropna(subset=['Tonnage'])
p0201_clean = p0201_clean[(p0201_clean['Year'] >= 2009) & (p0201_clean['Year'] <= 2025)]

# --- Prepare PORT0502 ---
def prep_0502_all_ports(df, direction_label):
    df_clean = df.dropna(subset=['Major Port']).copy()

    # Keep only the quarterly columns (those containing 'Q')
    quarter_cols = [col for col in df_clean.columns if 'Q' in str(col) and 'Percentage' not in str(col) and 'Four quarter' not in str(col)]

    # Keep ALL ports; no filtering through FOCUS_PORTS
    df_focus = df_clean[['Major Port'] + quarter_cols]

    # Reshape from wide to long format
    df_long = pd.melt(df_focus, id_vars=['Major Port'], value_vars=quarter_cols,
                      var_name='Quarter', value_name='Tonnage')
    df_long['Direction'] = direction_label
    df_long['Tonnage'] = num(df_long['Tonnage'])
    df_long = df_long.dropna(subset=['Tonnage'])

    # Extract the year so the period filter can be applied
    df_long['Quarter_Clean'] = df_long['Quarter'].astype(str).str.replace(r'\[.*\]', '', regex=True).str.strip()
    df_long['Year'] = df_long['Quarter_Clean'].str[:4].astype(int)
    df_long = df_long[(df_long['Year'] >= 2009) & (df_long['Year'] <= 2025)]

    # Convert the quarter label into a datetime value
    df_long['Date'] = pd.to_datetime(df_long['Quarter_Clean'].str[:4] + '-' +
                                     (df_long['Quarter_Clean'].str[-1].astype(int) * 3 - 2).astype(str).str.zfill(2) + '-01')
    return df_long.drop(columns=['Year', 'Quarter_Clean'])

p0502_clean = prep_0502_all_ports(p0502_raw, 'Both Directions')

# Remove the aggregate row, which would otherwise be counted as an additional port
p0502_clean = p0502_clean[p0502_clean['Major Port'] != 'Total at all major ports'].copy()

# ============================================================
# PART 3: REPORT THE CLEANED OUTPUT
# ============================================================
print(f"PORT0301 cleaned: {p0301_clean.shape[0]} observations ({p0301_clean['Year'].min()} to {p0301_clean['Year'].max()})")
print(f"PORT0201 cleaned: {p0201_clean.shape[0]} observations ({p0201_clean['Year'].min()} to {p0201_clean['Year'].max()})")
print(f"PORT0502 cleaned: {p0502_clean.shape[0]} observations (covering all {p0502_clean['Major Port'].nunique()} ports)")


PART 2: DATA PREPARATION
PORT0301 cleaned: 28513 observations (2009 to 2025)
PORT0201 cleaned: 2725 observations (2009 to 2025)
PORT0502 cleaned: 3604 observations (covering all 53 ports)


In [23]:
# ============================================================
# EXPORT THE CLEANED TABLES TO A SINGLE EXCEL WORKBOOK
# ============================================================
output_path = f'{OUT_DIR}/Cleaned_Maritime_Data_2009_2025.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    p0301_clean.to_excel(writer, sheet_name='PORT0301', index=False)
    p0201_clean.to_excel(writer, sheet_name='PORT0201', index=False)
    p0502_clean.to_excel(writer, sheet_name='PORT0502_Both', index=False)

print("Saved to:", output_path)
print("Number of ports in PORT0502:", p0502_clean['Major Port'].nunique())   # expected: 53

Saved to: /Users/rachelnguyen/thesis/outputs/Cleaned_Maritime_Data_2009_2025.xlsx
Number of ports in PORT0502: 53


In [24]:
# ===========================================================

# Work on a copy of the cleaned PORT0301 data
panel = p0301_clean.copy()

# Rename the column to the name used throughout the analysis
panel = panel.rename(columns={'Port Name': 'PortName'})
panel = panel[panel['Direction'] == 'Both Directions'].copy()

# Classify each record as Ro-Ro or Lo-Lo from the cargo group name
def get_mode(cargo_name):
    if 'Ro-Ro' in str(cargo_name):
        return 'Ro-Ro'
    elif 'Lo-Lo' in str(cargo_name):
        return 'Lo-Lo'
    else:
        return 'Other'

panel['Mode'] = panel['Cargo Group Name'].apply(get_mode)

# Keep only the Ro-Ro and Lo-Lo records
panel = panel[panel['Mode'] != 'Other']

# Aggregate the cargo sub-codes so that each port, mode and year has a single row
panel = panel.groupby(['Year', 'PortName', 'Mode'], as_index=False)['Tonnage'].sum()

# ============================================================
# DESCRIPTIVE STATISTICS
# ============================================================

# --- Summary statistics, all ports, 2015 to 2024 ---
window = panel[(panel['Year'] >= 2015) & (panel['Year'] <= 2024)]
summary = window.groupby('Mode')['Tonnage'].agg(
    N='count', mean='mean', SD='std',
    median='median', min='min', max='max').round(1)
print('Summary statistics (2015-2024):')
print(summary)
summary.to_csv(f'{OUT_DIR}/summary_statistics.csv')

# --- National totals before and after Brexit ---
print('\nPre vs Post-Brexit by mode:')
for mode in ['Ro-Ro', 'Lo-Lo']:
    md = panel[panel['Mode'] == mode]
    # Average annual national tonnage over 2015 to 2019
    pre = md[md['Year'].between(2015, 2019)].groupby('Year')['Tonnage'].sum().mean()
    # Average annual national tonnage over 2021 to 2024
    post = md[md['Year'].between(2021, 2024)].groupby('Year')['Tonnage'].sum().mean()
    print(f'  {mode}: pre={pre:.1f}  post={post:.1f}  change={(post/pre-1)*100:+.1f}%')

# --- Percentage change before and after Brexit at each port (route diversion) ---
changes = []
for mode in ['Ro-Ro', 'Lo-Lo']:
    s = panel[panel['Mode'] == mode]
    pre = s[s['Year'].between(2015, 2019)].groupby('PortName')['Tonnage'].mean()
    post = s[s['Year'].between(2021, 2024)].groupby('PortName')['Tonnage'].mean()
    c = pd.DataFrame({'Mode': mode, 'pre_2015_19': pre, 'post_2021_24': post})
    c['pct_change'] = (c['post_2021_24'] / c['pre_2015_19'] - 1) * 100
    changes.append(c.reset_index())

mode_change = pd.concat(changes).round(1)
mode_change.to_csv(f'{OUT_DIR}/mode_change_by_port.csv', index=False)

# --- Ports with the largest falls and the largest gains in Ro-Ro ---
rr = mode_change[mode_change['Mode'] == 'Ro-Ro'].dropna().sort_values('pct_change')
print('\nLargest Ro-Ro declines and gains:')
print(pd.concat([rr.head(6), rr.tail(6)])[['PortName', 'pct_change']].to_string(index=False))

Summary statistics (2015-2024):
         N    mean      SD  median  min      max
Mode                                            
Lo-Lo  197  3188.5  5469.0   912.2  0.0  25346.3
Ro-Ro  286  3589.4  5022.6  2054.9  0.1  27084.3

Pre vs Post-Brexit by mode:
  Ro-Ro: pre=105824.7  post=99418.0  change=-6.1%
  Lo-Lo: pre=65558.9  post=59745.6  change=-8.9%

Largest Ro-Ro declines and gains:
         PortName  pct_change
         Plymouth       -74.6
         Ramsgate       -71.8
Tees & Hartlepool       -40.5
             Hull       -40.4
         Newhaven       -40.3
             Tyne       -38.1
          Heysham        12.9
      Warrenpoint        13.2
          Belfast        18.1
           London        20.7
            Larne        35.0
        Cairnryan        35.0


In [25]:
pre_years  = [2015, 2016, 2017, 2018, 2019]
post_years = [2021, 2022, 2023, 2024]

pre = (panel[panel['Year'].isin(pre_years)]
       .groupby(['PortName','Mode'], as_index=False)['Tonnage'].mean()
       .rename(columns={'Tonnage':'pre_mean'}))

post = (panel[panel['Year'].isin(post_years)]
        .groupby(['PortName','Mode'], as_index=False)['Tonnage'].mean()
        .rename(columns={'Tonnage':'post_mean'}))

base = pre.merge(post, on=['PortName','Mode'], how='inner')
base['pct_change'] = (base['post_mean'] / base['pre_mean'] - 1) * 100

# Lo-Lo ports ranked by size rather than by percentage change,
# so that changes at very small ports do not dominate the ranking
ll = base[base['Mode'] == 'Lo-Lo'].sort_values('pre_mean', ascending=False).copy()
ll['share_pre_pct'] = ll['pre_mean'] / ll['pre_mean'].sum() * 100

print("Lo-Lo ports ranked by pre-Brexit tonnage:")
print(ll[['PortName','pre_mean','post_mean','pct_change','share_pre_pct']]
      .round(1).head(15).to_string(index=False))

ll.round(2).to_csv(f'{OUT_DIR}/lolo_route_diversion.csv', index=False)

Lo-Lo ports ranked by pre-Brexit tonnage:
           PortName  pre_mean  post_mean  pct_change  share_pre_pct
         Felixstowe   24205.1    18615.3       -23.1           37.0
             London   11979.7    14281.8        19.2           18.3
        Southampton    9925.1     8458.2       -14.8           15.2
          Liverpool    5589.8     5318.9        -4.8            8.5
  Tees & Hartlepool    2445.5     2361.6        -3.4            3.7
              Forth    2273.7     2118.3        -6.8            3.5
Grimsby & Immingham    2045.4     2146.6         4.9            3.1
               Hull    1775.9     2058.5        15.9            2.7
            Belfast    1688.6     1750.1         3.6            2.6
             Medway     930.0      484.9       -47.9            1.4
            Bristol     885.7      980.0        10.6            1.4
              Clyde     624.3      506.7       -18.8            1.0
               Tyne     404.0      295.4       -26.9            0.6
      

In [26]:
# ============================================================
# ROUTE DIVERSION MAP DATA
# Builds the file used by the Power BI maps for Ro-Ro and Lo-Lo.
# Run after the cell that creates 'mode_change'.
# ============================================================

import numpy as np
import pandas as pd

# ---------- 1. Percentage change by port and mode ----------
route_map = (mode_change
             .dropna(subset=['pct_change'])
             .rename(columns={'pre_2015_19': 'pre_mean',
                              'post_2021_24': 'post_mean'})
             .copy())

# Share of each port in the total tonnage of its own mode
route_map['share_pre_pct'] = (route_map['pre_mean'] /
                              route_map.groupby('Mode')['pre_mean'].transform('sum') * 100)

route_map['direction'] = np.where(route_map['pct_change'] > 0, 'Increase', 'Decrease')
route_map['abs_pct']   = route_map['pct_change'].abs()

# Country column, so that Power BI resolves each port name within the United Kingdom
route_map['Country'] = 'United Kingdom'

# Tooltip label, for example "Plymouth: -74.6%"
route_map['label'] = (route_map['PortName'] + ': ' +
                      route_map['pct_change'].round(1).astype(str) + '%')

# ---------- 2. Tidy and export ----------
route_map = route_map.sort_values(['Mode', 'pct_change']).round(2)
route_map = route_map[['PortName', 'Country', 'Mode',
                       'pre_mean', 'post_mean', 'pct_change', 'abs_pct',
                       'direction', 'share_pre_pct', 'label']]

route_map.to_csv(f'{OUT_DIR}/route_map_combined.csv', index=False, encoding='utf-8-sig')

# ---------- 3. Checks ----------
print("Number of ports in each mode:")
print(route_map.groupby('Mode').size().to_string())

print("\nRange of pct_change by mode:")
print(route_map.groupby('Mode')['pct_change'].agg(['min','max']).round(1).to_string())

print("\nSum of share_pre_pct by mode (should be 100):")
print(route_map.groupby('Mode')['share_pre_pct'].sum().round(1).to_string())

print(f"\nSaved to: {OUT_DIR}/route_map_combined.csv  ({len(route_map)} rows)")
route_map.head()

Number of ports in each mode:
Mode
Lo-Lo    19
Ro-Ro    28

Range of pct_change by mode:
        min   max
Mode             
Lo-Lo -95.4  19.2
Ro-Ro -74.6  35.0

Sum of share_pre_pct by mode (should be 100):
Mode
Lo-Lo    100.0
Ro-Ro    100.0

Saved to: /Users/rachelnguyen/thesis/outputs/route_map_combined.csv  (47 rows)


,PortName,Country,Mode,pre_mean,post_mean,pct_change,abs_pct,direction,share_pre_pct,label
15,Manchester,United Kingdom,Lo-Lo,1.5,0.1,-95.4,95.4,Decrease,0.00,Manchester: -95.4%
0,Aberdeen,United Kingdom,Lo-Lo,80.1,34.8,-56.5,56.5,Decrease,0.12,Aberdeen: -56.5%
10,Harwich,United Kingdom,Lo-Lo,20.9,10.5,-49.8,49.8,Decrease,0.03,Harwich: -49.8%
16,Medway,United Kingdom,Lo-Lo,930.0,484.9,-47.9,47.9,Decrease,1.42,Medway: -47.9%
24,Warrenpoint,United Kingdom,Lo-Lo,269.3,155.7,-42.2,42.2,Decrease,0.41,Warrenpoint: -42.2%


In [27]:
# ============================================================
# REGRESSION
# ============================================================
def report(model, label):
    """Print effect sizes (percent) for the key regression terms."""
    print(f'--- {label} | R2={model.rsquared:.3f} n={int(model.nobs)} ---')
    for t in ['post_brexit', 'post_brexit:roro', 'covid']:
        if t in model.params:
            b, p_ = model.params[t], model.pvalues[t]
            print(f'  {t:>18}: {100*(np.exp(b)-1):+.1f}%  (p={p_:.4f})')

# ------------------------------------------------------------
# STEP 1: CREATE THE VARIABLES ON THE PANEL DATASET
# ------------------------------------------------------------
panel['cell'] = panel['PortName'] + '_' + panel['Mode']   # port and mode combination
panel = panel[panel['Tonnage'] > 0].copy()                # drop zeros before taking logs
panel['log_tonnage'] = np.log(panel['Tonnage'])           # natural log, no constant added
panel['roro'] = (panel['Mode'] == 'Ro-Ro').astype(int)
panel['post_brexit'] = (panel['Year'] >= BREXIT_YEAR).astype(int)
panel['covid'] = (panel['Year'] == COVID_YEAR).astype(int)
panel['trend'] = panel['Year'] - 2000


# ------------------------------------------------------------
# STEP 2: BUILD THE NATIONAL DATASET FOR SPECIFICATION B
# ------------------------------------------------------------
# Map the PORT0201 cargo codes onto the two handling modes
def code_to_mode(code):
    code = int(code)
    if 51 <= code <= 59: return 'Ro-Ro'
    elif 31 <= code <= 34: return 'Lo-Lo'
    return 'Other'

nat = p0201_clean.copy()
nat['Mode'] = nat['CargoCode'].apply(code_to_mode)
nat = nat[nat['Mode'].isin(['Ro-Ro','Lo-Lo'])]
nat = nat[nat['Region'].astype(str).str.contains('International', case=False, na=False)]
nat = nat[nat['Tonnage'] > 0]

national = nat.groupby(['Year','Mode'])['Tonnage'].sum().reset_index()
national = national[(national['Year']>=2015)&(national['Year']<=2024)]
national['log_tonnage'] = np.log(national['Tonnage'])
national['roro'] = (national['Mode']=='Ro-Ro').astype(int)
national['post_brexit'] = (national['Year']>=BREXIT_YEAR).astype(int)
national['covid'] = (national['Year']==COVID_YEAR).astype(int)

# ------------------------------------------------------------
# STEP 3: ESTIMATE THE TWO SPECIFICATIONS
# ------------------------------------------------------------
# ---- Specification A: all-port panel ----
a = panel[(panel['Year']>=2015) & (panel['Year']<=2024)].copy()
cell_means = a.groupby('cell')['Tonnage'].mean()
a = a[a['cell'].isin(cell_means[cell_means >= 1000].index)]   # keep cells averaging at least 1 million tonnes
specA = smf.ols('log_tonnage ~ post_brexit * roro + covid + C(cell) + trend',
                data=a).fit(cov_type='HC3')
report(specA, 'Spec A: all-port panel (cells >= 1M tonnes)')

# ---- Specification B: national difference-in-differences, international traffic ----
specB = smf.ols('log_tonnage ~ post_brexit * roro + covid', data=national).fit(cov_type='HC3')
report(specB, 'Spec B: national international DiD')

# ------------------------------------------------------------
# STEP 4: ROBUSTNESS CHECKS
# ------------------------------------------------------------
rob_rows = []

def robust(df, label, weights=None):
    """Run one robustness variant with cell-clustered standard errors."""
    kwargs = dict(cov_type='cluster', cov_kwds={'groups': df['cell']})
    formula = 'log_tonnage ~ post_brexit * roro + covid + C(cell) + trend'
    if weights is not None:
        m = smf.wls(formula, data=df, weights=weights).fit(**kwargs)
    else:
        m = smf.ols(formula, data=df).fit(**kwargs)

    b, p_ = m.params['post_brexit:roro'], m.pvalues['post_brexit:roro']
    rob_rows.append([label, int(m.nobs), df['cell'].nunique(),
                     f'{100*(np.exp(b)-1):+.1f}%', round(p_, 4)])

# Restrict to the larger cells
big = panel[panel['cell'].isin(panel.groupby('cell')['Tonnage'].mean().pipe(lambda s: s[s >= 500]).index)].copy()

robust(big, 'P1: all ports, cells >= 0.5M')
robust(big[big['cell'].isin(big.groupby('cell')['Tonnage'].mean().pipe(lambda s: s[s >= 1000]).index)],
       'P2: all ports, cells >= 1M')
robust(big, 'P3: WLS tonnage-weighted',
       weights=big.groupby('cell')['Tonnage'].transform('mean'))
robust(big[(big['Year']>=2015)&(big['Year']<=2024)], 'P4: 2015-2024 window')

robustness = pd.DataFrame(rob_rows, columns=['Model', 'n', 'cells', 'BrexitxRoRo', 'p'])
robustness.to_csv(f'{OUT_DIR}/robustness_table.csv', index=False)
print('\nRobustness table (interaction term, cluster SEs):')
print(robustness.to_string(index=False))

with open(f'{OUT_DIR}/regression_full_output.txt', 'w') as f:
    f.write('=== SPEC A ===\n' + str(specA.summary())
            + '\n\n=== SPEC B ===\n' + str(specB.summary()))

print("national shape:", national.shape)
print(national)

--- Spec A: all-port panel (cells >= 1M tonnes) | R2=0.974 n=270 ---
         post_brexit: -8.8%  (p=0.0828)
    post_brexit:roro: -1.3%  (p=0.7445)
               covid: -6.7%  (p=0.0498)
--- Spec B: national international DiD | R2=0.983 n=20 ---
         post_brexit: -6.7%  (p=0.0066)
    post_brexit:roro: -21.7%  (p=0.0000)
               covid: -10.6%  (p=0.1735)

Robustness table (interaction term, cluster SEs):
                       Model   n  cells BrexitxRoRo      p
P1: all ports, cells >= 0.5M 573     36       -4.3% 0.7305
  P2: all ports, cells >= 1M 461     29       -0.6% 0.9666
    P3: WLS tonnage-weighted 573     36       -2.3% 0.8835
        P4: 2015-2024 window 334     34       -2.3% 0.7676
national shape: (20, 7)
    Year   Mode        Tonnage  log_tonnage  roro  post_brexit  covid
12  2015  Lo-Lo  118068.678961    11.679022     0            0      0
13  2015  Ro-Ro   90870.828437    11.417194     1            0      0
14  2016  Lo-Lo  123183.774623    11.721433     0 

In [28]:
# ============================================================
# TOTAL QUARTERLY (dữ liệu đã bỏ dòng tổng)
# ============================================================
total_q = (p0502_clean.groupby('Date', as_index=False)['Tonnage'].sum()
           .rename(columns={'Date': 'ds', 'Tonnage': 'y'}))
total_q['ds'] = pd.to_datetime(total_q['ds'])
total_q = total_q.sort_values('ds').reset_index(drop=True)

print("Số quý:", len(total_q))
print("Tổng 4 quý cuối:", round(total_q['y'].tail(4).sum()/1000,1), "triệu tấn")  # phải ~421
print(total_q.tail(6).to_string(index=False))

Số quý: 68
Tổng 4 quý cuối: 419.5 triệu tấn
        ds             y
2024-07-01 102987.590000
2024-10-01 107894.890000
2025-01-01 107546.924340
2025-04-01 106162.667319
2025-07-01 102935.382894
2025-10-01 102894.259536


In [29]:
# Check that total_q is correct before forecasting
print("Check: sum of the last four quarters =", round(total_q['y'].tail(4).sum()/1000,1), "million tonnes (expected around 421)")
print("Most recent quarter =", round(total_q['y'].iloc[-1]/1000,1), "million tonnes (expected around 105)")

# ============================================================
# FORECAST TOTAL VOLUME: Prophet vs ARIMA
# ============================================================
from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX
import itertools

train = total_q[total_q['ds'] <= '2023-10-01'].copy()
test  = total_q[total_q['ds'] >  '2023-10-01'].copy()
print("Train:", len(train), "quarters | Test:", len(test), "quarters")

# --- Prophet ---
mp = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
mp.fit(train.rename(columns={'ds':'ds','y':'y'}))
future = mp.make_future_dataframe(periods=len(test), freq='QS')
prophet_pred = mp.predict(future).tail(len(test))['yhat'].values

# --- ARIMA grid search on the Akaike Information Criterion ---
best_aic, best_fit, best_order = np.inf, None, None
for order in itertools.product(range(3),range(2),range(3),range(2),range(2),range(2)):
    p_,d_,q_,P_,D_,Q_ = order
    try:
        r = SARIMAX(train['y'], order=(p_,d_,q_), seasonal_order=(P_,D_,Q_,4),
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=0)
        if r.aic < best_aic: best_aic, best_fit, best_order = r.aic, r, (p_,d_,q_,P_,D_,Q_)
    except: continue

if best_fit is None:
    raise ValueError("ARIMA grid search found no valid model, check the training data")

arima_pred = best_fit.forecast(len(test)).values

# --- Accuracy measures: MAE, RMSE, MAPE ---
from sklearn.metrics import mean_absolute_error, mean_squared_error

def mape(a,f): return np.mean(np.abs((a-f)/a))*100

actual = test['y'].values

print("\n=== FORECAST ACCURACY (test period 2024-2025) ===")
for name, pred in [('Prophet', prophet_pred), ('ARIMA', arima_pred)]:
    mae_val  = mean_absolute_error(actual, pred)
    rmse_val = np.sqrt(mean_squared_error(actual, pred))
    mape_val = mape(actual, pred)
    print(f"{name:8s}  MAE={mae_val:8.1f}  RMSE={rmse_val:8.1f}  MAPE={mape_val:.2f}%")

print(f"\nARIMA order: {best_order}")

# Save the accuracy table for the dissertation
metrics_df = pd.DataFrame([
    {'Model':'Prophet',
     'MAE':  round(mean_absolute_error(actual, prophet_pred),1),
     'RMSE': round(np.sqrt(mean_squared_error(actual, prophet_pred)),1),
     'MAPE': round(mape(actual, prophet_pred),2)},
    {'Model':'ARIMA',
     'MAE':  round(mean_absolute_error(actual, arima_pred),1),
     'RMSE': round(np.sqrt(mean_squared_error(actual, arima_pred)),1),
     'MAPE': round(mape(actual, arima_pred),2)},
])
metrics_df.to_csv(f'{OUT_DIR}/forecast_metrics.csv', index=False)
print("\nSaved forecast_metrics.csv:")
print(metrics_df.to_string(index=False))

Check: sum of the last four quarters = 419.5 million tonnes (expected around 421)
Most recent quarter = 102.9 million tonnes (expected around 105)
Train: 60 quarters | Test: 8 quarters


01:06:21 - cmdstanpy - INFO - Chain [1] start processing
01:06:21 - cmdstanpy - INFO - Chain [1] done processing



=== FORECAST ACCURACY (test period 2024-2025) ===
Prophet   MAE=  3553.8  RMSE=  4560.2  MAPE=3.33%
ARIMA     MAE=  7581.3  RMSE=  8161.6  MAPE=7.19%

ARIMA order: (2, 1, 2, 1, 1, 1)

Saved forecast_metrics.csv:
  Model    MAE   RMSE  MAPE
Prophet 3553.8 4560.2  3.33
  ARIMA 7581.3 8161.6  7.19


In [30]:
# ============================================================
# FORECAST 2026
# ============================================================
mp_full = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
mp_full.fit(total_q.rename(columns={'ds':'ds','y':'y'}))
future_2026 = mp_full.make_future_dataframe(periods=4, freq='QS')
fc_2026 = mp_full.predict(future_2026).tail(4)[['ds','yhat']]
fc_2026['Quarter'] = ['2026 Q1','2026 Q2','2026 Q3','2026 Q4']
fc_2026['Forecast'] = fc_2026['yhat'].round(1)
print("\nForecast 2026 (Prophet):")
print(fc_2026[['Quarter','Forecast']].to_string(index=False))

# ---  actual vs predicted (test window)
compare = test[['ds','y']].copy()
compare['Prophet'] = prophet_pred.round(1)
compare['ARIMA'] = arima_pred.round(1)
compare['Quarter'] = compare['ds'].dt.year.astype(str) + ' Q' + compare['ds'].dt.quarter.astype(str)
compare = compare.rename(columns={'y':'Actual'})
print("\nActual vs Predicted (test window):")
print(compare[['Quarter','Actual','Prophet','ARIMA']].to_string(index=False))

# --- Export CSV for Power BI ---
compare[['Quarter','Actual','Prophet','ARIMA']].to_csv(f'{OUT_DIR}/forecast_test.csv', index=False)
fc_2026[['Quarter','Forecast']].to_csv(f'{OUT_DIR}/forecast_2026.csv', index=False)
print("\n saved forecast_test.csv và forecast_2026.csv")

01:06:27 - cmdstanpy - INFO - Chain [1] start processing
01:06:27 - cmdstanpy - INFO - Chain [1] done processing



Forecast 2026 (Prophet):
Quarter  Forecast
2026 Q1  103489.0
2026 Q2  104861.6
2026 Q3  103274.3
2026 Q4  104533.4

Actual vs Predicted (test window):
Quarter        Actual  Prophet    ARIMA
2024 Q1 102476.252000 105734.8 104887.5
2024 Q2 107657.977000 101312.3 100179.8
2024 Q3 102987.590000 102646.1  96613.9
2024 Q4 107894.890000 101147.0 101860.5
2025 Q1 107546.924340  99946.4  98055.4
2025 Q2 106162.667319 102731.0  92727.3
2025 Q3 102935.382894 102446.6  93714.6
2025 Q4 102894.259536 102678.5  96689.2

 saved forecast_test.csv và forecast_2026.csv


In [31]:
# ============================================================
# BUILD THE CSV FILES FOR THE POWER BI DASHBOARD
# ============================================================

# --- total volume by year, from PORT0301, all ports ---
kpi1 = (p0301_clean[
    (p0301_clean['Cargo Group Name'] == 'All Cargo') &
    (p0301_clean['Direction'] == 'Both Directions')]
    .groupby('Year', as_index=False)['Tonnage'].sum())
kpi1['Tonnage_Million'] = (kpi1['Tonnage'] / 1000).round(1)
kpi1.to_csv(f'{OUT_DIR}/kpi1_total_volume.csv', index=False)
print("KPI 1 (total volume by year):", kpi1.shape)
print(kpi1[['Year','Tonnage_Million']].to_string(index=False))

# --- Ro-Ro and Lo-Lo by year, from the panel ---
kpi2 = panel.groupby(['Year','Mode'], as_index=False)['Tonnage'].sum()
kpi2['Tonnage_Million'] = (kpi2['Tonnage'] / 1000).round(1)
kpi2.to_csv(f'{OUT_DIR}/kpi2_mode_split.csv', index=False)
print("\nKPI 2 (Ro-Ro and Lo-Lo by year):", kpi2.shape)

# --- route diversion, Ro-Ro only ---
kpi3 = mode_change[mode_change['Mode']=='Ro-Ro'].dropna(subset=['pct_change']).copy()
kpi3.to_csv(f'{OUT_DIR}/kpi3_route_map.csv', index=False)
print("\nKPI 3 (route diversion):", kpi3.shape)

KPI 1 (total volume by year): (17, 3)
 Year  Tonnage_Million
 2009            489.6
 2010            498.5
 2011            507.0
 2012            489.5
 2013            491.8
 2014            491.9
 2015            485.7
 2016            472.8
 2017            470.7
 2018            472.1
 2019            471.7
 2020            429.0
 2021            435.4
 2022            449.7
 2023            425.9
 2024            421.0
 2025            419.5

KPI 2 (Ro-Ro and Lo-Lo by year): (34, 4)

KPI 3 (route diversion): (28, 5)


In [32]:
# ============================================================
# BUILD THE FORECAST FILES FOR POWER BI
# ============================================================
# --- 1. Model comparison over the test window (2024-2025), for Power BI ---
compare = test[['ds','y']].copy()
compare['Prophet'] = prophet_pred.round(1)
compare['ARIMA']   = arima_pred.round(1)
compare['Period']  = compare['ds'].dt.year.astype(str) + ' Quarter ' + compare['ds'].dt.quarter.astype(str)
compare = compare.rename(columns={'y':'Actual'})
compare[['Period','Actual','Prophet','ARIMA']].to_csv(f'{OUT_DIR}/powerbi_model_comparison.csv', index=False)
print("\nSaved powerbi_model_comparison.csv")
print(compare[['Period','Actual','Prophet','ARIMA']].to_string(index=False))

# --- 2. Forecast for 2026, using Prophet trained on the full series to 2025 ---
mp_full = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
mp_full.fit(total_q.rename(columns={'ds':'ds','y':'y'}))
future_2026 = mp_full.make_future_dataframe(periods=4, freq='QS')
fc = mp_full.predict(future_2026).tail(4)

# --- 3. Combine history and forecast into a single long format file ---
rows = []

# Observed quarters
for _, r in total_q.iterrows():
    d = r['ds']
    rows.append({
        'Period': f"{d.year} Quarter {(d.month-1)//3+1}",
        'Type': 'Actual',
        'Tonnage': round(r['y'], 1),
        'PeriodOrder': d.year*10 + (d.month-1)//3+1
    })

# Forecast quarters for 2026
for _, r in fc.iterrows():
    d = r['ds']
    rows.append({
        'Period': f"{d.year} Quarter {(d.month-1)//3+1}",
        'Type': 'Forecast',
        'Tonnage': round(r['yhat'], 1),
        'PeriodOrder': d.year*10 + (d.month-1)//3+1
    })

powerbi_forecast = pd.DataFrame(rows).sort_values('PeriodOrder').reset_index(drop=True)
powerbi_forecast.to_csv(f'{OUT_DIR}/powerbi_total_forecast.csv', index=False)
print("Saved powerbi_total_forecast.csv:", powerbi_forecast.shape)
print("\nLast rows, showing the forecast continuing from the observed series:")
print(powerbi_forecast.tail(8).to_string(index=False))

01:06:27 - cmdstanpy - INFO - Chain [1] start processing



Saved powerbi_model_comparison.csv
        Period        Actual  Prophet    ARIMA
2024 Quarter 1 102476.252000 105734.8 104887.5
2024 Quarter 2 107657.977000 101312.3 100179.8
2024 Quarter 3 102987.590000 102646.1  96613.9
2024 Quarter 4 107894.890000 101147.0 101860.5
2025 Quarter 1 107546.924340  99946.4  98055.4
2025 Quarter 2 106162.667319 102731.0  92727.3
2025 Quarter 3 102935.382894 102446.6  93714.6
2025 Quarter 4 102894.259536 102678.5  96689.2


01:06:28 - cmdstanpy - INFO - Chain [1] done processing


Saved powerbi_total_forecast.csv: (72, 4)

Last rows, showing the forecast continuing from the observed series:
        Period     Type  Tonnage  PeriodOrder
2025 Quarter 1   Actual 107546.9        20251
2025 Quarter 2   Actual 106162.7        20252
2025 Quarter 3   Actual 102935.4        20253
2025 Quarter 4   Actual 102894.3        20254
2026 Quarter 1 Forecast 103489.0        20261
2026 Quarter 2 Forecast 104861.6        20262
2026 Quarter 3 Forecast 103274.3        20263
2026 Quarter 4 Forecast 104533.4        20264


In [33]:
import os, glob
print("OUT_DIR:", OUT_DIR)
files = sorted(glob.glob(os.path.join(OUT_DIR, '*.csv')))
print("Number of CSV files:", len(files))
for f in files:
    print("  ", os.path.basename(f))

OUT_DIR: /Users/rachelnguyen/thesis/outputs
Number of CSV files: 13
   forecast_2026.csv
   forecast_metrics.csv
   forecast_test.csv
   kpi1_total_volume.csv
   kpi2_mode_split.csv
   kpi3_route_map.csv
   lolo_route_diversion.csv
   mode_change_by_port.csv
   powerbi_model_comparison.csv
   powerbi_total_forecast.csv
   robustness_table.csv
   route_map_combined.csv
   summary_statistics.csv
